# Fock-G1: First-Order Ablation of Aniso-Gaussian Fock-PARFLM — OpenWebText d=384 (Phase 1)

## What this is

The **Fock-G1** first-order ablation defined in §6 of
[`Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md`](../../../companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md),
here applied to the **d=384 OpenWebText primary arm** (§6.2 of that protocol).
It is the large-scale/OpenWebText analogue of the TinyStories Fock-G1 run
(`first_order_ablation/colab_fock_g1_aniso_gaussian_fockreg_tinystories.ipynb`).

It answers the **training-phase** question at d=384: does the inertial
(velocity) term contribute genuine value *during training*, at matched compute
and matched step size — or does a well-chosen but fixed step size fully recover
the second-order winner?

## Phase 1: first 100K steps

This is **Phase 1** of the d=384 experiment — the first 100K steps under the WSD
schedule, matching the second-order anchor's own Phase-1 notebook. A
continuation notebook resumes for the remaining steps. The H1/H0/H-1 verdict is
read at full convergence, not at the Phase-1 boundary; the 100K checkpoint here
is an interim comparison point.

## The single architectural change

The second-order Fock v2.1 layer update is a damped velocity-Verlet step
(`model_parf_multixi.py::_layer_step`):

    delta = h - h_prev
    denom = 1 + dt * gamma
    h_new = LN( h + delta/denom + dt^2/(m_b*denom) * f )     # second order

Fock-G1 forces the velocity memory `delta = h - h_prev` to **zero** at every
layer, collapsing the update to a pure first-order gradient step:

    h_new = LN( h + beta * f ),   beta = dt^2 / (m_b * (1 + dt * gamma*))

Because gamma is fixed at gamma* (`FIXED_GAMMA = 0.10`, the d=384 sweep winner),
beta is a **fixed constant** equal to the second-order anchor's own initial
effective step size — so the ablation isolates the inertial term and nothing
else. gamma* is **not** a damping coefficient here (there is no velocity to
damp); it only pins the step size (see §6.1 of the protocol).

**Everything else is inherited unchanged** from the second-order arm: the
anisotropic Gaussian V_theta (diag + low-rank precision), the pairwise V_phi
(structural_competitive), the multi-channel xi context, the Fock register
creation/destruction gates, the reverse channel (E5c stabilisation), the Fock
coupling regularisation, per-group grad clipping, the WSD schedule, and every
other hyperparameter. Per the SPLM-1 design discipline, **no independent LR
sweep is performed**.

## Causal-leak checks (same as the second-order run)

Both leak checks from the second-order notebook run unchanged and on the same
schedule (every 10K steps): the lightweight **architectural causal probe**
(future-token insensitivity; `max|dlogit|` must be 0) — instantiated on the
**Fock-G1 model class** so it tests exactly what is being trained — and the
**trained leak probe** (`probe_trained_leak` + `honest_ppl_test`) on the live
model.

## Comparison anchor and decision rule (OWT, d=384)

The second-order anchor is the aniso-Gaussian + fock-reg d=384 OWT run at
**gamma* = 0.10** (§12 of
[`Determining_optimal_gamma_for_Fock-PARFLM.md`](../../../companion_notes/Determining_optimal_gamma_for_Fock-PARFLM.md)),
sweep-best **PPL ≈ 278.27**.

| Hypothesis | Operational form | Reading |
|---|---|---|
| H1 (training-time value) | mean Δ ≥ Δmin | inertia adds genuine value at matched compute and step size |
| H0 (artefact) | \|mean Δ\| < Δmin | the interior gamma* is an effective-step-size artefact; Fock-G1 matches |
| H-1 (refutation) | mean Δ ≤ −Δmin | Fock-G1 outperforms; training-time-value claim falsified for this family |

with Δ = PPL(Fock-G1) − PPL(second-order anchor), averaged over seeds.
**Δmin = 15.0 PPL** for the d=384 primary arm (§6.4 of the protocol: the larger
adjacent-in-the-bowl gap is 292.37 − 278.27 = 14.10 at gamma 0.10→0.25, and the
SPLM-1-proportional estimate is ~16). The full protocol runs **3 seeds**
(SEED = 0, 1, 2); run this notebook once per seed. Because this is only Phase 1
(100K of the full schedule), read the endpoint here as an interim datapoint, not
the final verdict.

## Companion documents

- `companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md` (§6)
- `companion_notes/Determining_optimal_gamma_for_Fock-PARFLM.md` (§12)
- `companion_notes/SPLM-1_ablation_pre-registered_protocol.md`


In [ ]:
# == Cell 0: Configuration =============================================

# -- First-order ablation (Fock-G1) ------------------------------------
FIRST_ORDER = True     # zero velocity memory (delta := 0) => pure gradient step

# -- V_theta: Anisotropic Gaussian (diagonal + low-rank precision) -----
V_THETA_VARIANT             = 'aniso_gaussian'
V_THETA_WELLS_PER_HEAD      = 8
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02
ANISO_RANK                  = 4
W_SCALE                     = 1.0

# -- Xi channels (5long preset from OWT notebook) ----------------------
XI_OVERRIDE     = '5long'
_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
XI_CHANNELS    = len(XI_ALPHA_INITS)
V_THETA_N_HEADS = XI_CHANNELS

# -- PARF V_phi --------------------------------------------------------
V_PHI_KIND      = 'structural_competitive'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32
V_PHI_D_ANGLE   = 16

# -- Reverse channel stabilisation (E5c) -------------------------------
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True
REVERSE_CHANNEL_RESET_SCALE  = False

# -- Register repulsion (B4) -------------------------------------------
REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05
REGISTER_REPULSION_KIND  = 'gram'

# -- Output head -------------------------------------------------------
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False

# -- Optimizer ---------------------------------------------------------
OPTIMIZER = 'adamw'
GRAD_CENTRALIZATION = False

# -- LR schedule (WSD) -------------------------------------------------
LR_SCHEDULE     = 'wsd'
WSD_WARMUP_FRAC = 0.05
WSD_STABLE_FRAC = 0.60
WSD_LR_FLOOR    = None          # resolved after LR is set

# -- Batch / accumulation (explicit) -----------------------------------
GRAD_ACCUM      = 2

# -- Fock coupling regularisation --------------------------------------
LAMBDA_FOCK_REG = 5e-3
FOCK_REG_EPS    = 1e-6

# -- Damping coefficient -------------------------------------------------
# gamma=0.100 chosen from the d=384, L=16 aniso-Gaussian+fock-reg gamma
# sweep (colab_fock_gamma_sweep_geodesic_aniso_gaussian_fockreg_d384.ipynb):
# best PPL (278.27) AND best geodesic R_bar (0.6708) coincide at gamma=0.100.
# gamma=0.150/0.250 are tied within ~5% (flat bowl); gamma=0.200 was an
# isolated instability outlier (PPL=2250) bracketed by good neighbours on
# both sides, not a genuine stability wall.
FIXED_GAMMA = 0.10
GAMMA_STAR  = FIXED_GAMMA   # alias; gamma has no dynamical role in first order

# -- Second-order comparison anchor (Determining_optimal_gamma §12) ------
SECOND_ORDER_ANCHOR_PPL = 278.27   # aniso-Gaussian + fock-reg d=384 OWT, gamma*=0.10
DELTA_MIN_PPL           = 15.0     # decision threshold (protocol §6.4)

# -- Regularisation ----------------------------------------------------
LAMBDA_V       = 1e-2
BG_QUAD_EPS    = 0.0

# -- Training ----------------------------------------------------------
TOTAL_STEPS   = 100_000
BLOCK_SIZE    = 512
VOCAB_SIZE    = 50257
SEED          = 0

# -- Variant tag (for GDrive path and checkpoint naming) ----------------
_variant_parts = []
_variant_parts.append(f'xi{XI_OVERRIDE}')
_variant_parts.append(f'topk{TOP_K}')
_variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
_variant_parts.append(f'mh{V_PHI_N_HEADS}')
_variant_parts.append(f'aniso_dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
_variant_parts.append('ob')
_variant_parts.append('untied')
_variant_parts.append(LR_SCHEDULE)
_variant_parts.append('e5c')
_variant_parts.append('plgate')
_variant_parts.append(f'rep{REGISTER_REPULSION_COEFF:g}')
_variant_parts.append(f'fockreg{LAMBDA_FOCK_REG:g}')
_variant_parts.append(f'g{FIXED_GAMMA:g}')
_variant_parts.append('g1fo')   # Fock-G1 first-order marker (keeps paths separate)
_variant_tag = '_'.join(_variant_parts)

total_wells = V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD
print(f'Fock-G1 FIRST-ORDER ablation — Aniso-Gaussian on OpenWebText d=384 (Phase 1)')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{total_wells} total attractors')
print(f'  Aniso rank r={ANISO_RANK}')
print(f'  Depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  Fock coupling reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  first_order={FIRST_ORDER}  gamma*={FIXED_GAMMA} (fixed; defines beta, no inertia)')
print(f'  2nd-order anchor PPL={SECOND_ORDER_ANCHOR_PPL}  Delta_min={DELTA_MIN_PPL}')
print(f'  Xi: {XI_CHANNELS}ch  horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS}h  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  steps={TOTAL_STEPS}  schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
print(f'  [variant] tag={_variant_tag}')

In [ ]:
# == Cell 1: Environment + Drive Mount =================================
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_g1_aniso_gaussian_fockreg_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'fock_g1_aniso_gaussian_fockreg_owt' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_g1_aniso_owt' + (f'_{_variant_tag}' if _variant_tag else '')
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# == Cell 2: GPU + Checkpoint resolution ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) -- resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'Resuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found -- training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# == Cell 3: Data loading (OpenWebText) ================================
from data_module import get_batch

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# == Cell 4: Anisotropic Gaussian V_theta ==============================
#
# Extends the base MixtureGaussianVTheta with low-rank cross-correlations.
#
# Precision per well:  Sigma_k^{-1} = diag(a_k) + B_k @ B_k^T
# where B_k in R^{d x r} is a learned low-rank factor.
#
# Forces remain bounded (same diff*exp(-r^2/2) decay).


class AnisotropicMixtureGaussianVTheta(nn.Module):

    def __init__(self, d: int, K: int = 8, rank: int = 4,
                 w_scale: float = 1.0, xi_d=None,
                 init_log_precision=None, precision_max=None,
                 force_norm_max=None):
        super().__init__()
        self.d = d
        self.K = K
        self.rank = rank
        self.w_scale = w_scale
        self._precision_max = precision_max
        self._force_norm_max = force_norm_max
        in_d = xi_d if xi_d is not None else d

        self.mu_proj = nn.Linear(in_d, K * d)
        self.a_proj = nn.Linear(in_d, K * d)
        self.w_proj = nn.Linear(in_d, K)
        self.B_proj = nn.Linear(in_d, K * d * rank)

        self._init_weights(init_log_precision)

    def _init_weights(self, init_log_precision):
        nn.init.xavier_uniform_(self.mu_proj.weight)
        nn.init.zeros_(self.mu_proj.bias)
        nn.init.zeros_(self.a_proj.weight)
        if init_log_precision is not None:
            self.a_proj.bias.data.fill_(init_log_precision)
        else:
            self.a_proj.bias.data.fill_(0.0)
        nn.init.zeros_(self.w_proj.weight)
        nn.init.zeros_(self.w_proj.bias)
        nn.init.normal_(self.B_proj.weight, std=0.01)
        nn.init.zeros_(self.B_proj.bias)

    def _components(self, xi):
        lead = xi.shape[:-1]
        mu = self.mu_proj(xi).view(*lead, self.K, self.d)
        a = (F.softplus(self.a_proj(xi)) + 1e-4).view(*lead, self.K, self.d)
        if self._precision_max is not None:
            a = a.clamp(max=self._precision_max)
        w = F.softmax(self.w_proj(xi), dim=-1) * self.w_scale
        B = self.B_proj(xi).view(*lead, self.K, self.d, self.rank)
        return mu, a, w, B

    def forward(self, xi, h):
        mu, a, w, B = self._components(xi)
        h_e = h.unsqueeze(-2)
        diff = h_e - mu
        diag_term = (a * diff * diff).sum(dim=-1)
        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
        exponent = -0.5 * (diag_term + lr_term)
        bumps = w * torch.exp(exponent)
        return -bumps.sum(dim=-1, keepdim=True)

    def analytical_grad(self, xi, h):
        mu, a, w, B = self._components(xi)
        h_e = h.unsqueeze(-2)
        diff = h_e - mu
        diag_term = (a * diff * diff).sum(dim=-1)
        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
        exponent = -0.5 * (diag_term + lr_term)
        g = w * torch.exp(exponent)
        grad_diag = a * diff
        grad_lr = torch.einsum('...kdr,...kr->...kd', B, Bt_diff)
        per_comp = (grad_diag + grad_lr) * g.unsqueeze(-1)
        if self._force_norm_max is not None:
            norms = per_comp.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            scale = (self._force_norm_max / norms).clamp(max=1.0)
            per_comp = per_comp * scale
        return per_comp.sum(dim=-2)

    def attractor_centres(self, xi):
        lead = xi.shape[:-1]
        return self.mu_proj(xi).view(*lead, self.K, self.d)


class AnisotropicMultiContextGaussianVTheta(nn.Module):

    def __init__(self, d, K, n_ctx, rank=4, w_scale=1.0,
                 init_log_precision=None, precision_max=None,
                 force_norm_max=None):
        super().__init__()
        self.d = d
        self.K = K
        self.n_ctx = n_ctx
        self.banks = nn.ModuleList(
            AnisotropicMixtureGaussianVTheta(
                d=d, K=K, rank=rank, w_scale=w_scale, xi_d=d,
                init_log_precision=init_log_precision,
                precision_max=precision_max,
                force_norm_max=force_norm_max,
            )
            for _ in range(n_ctx)
        )

    def forward(self, xis, h):
        out = self.banks[0](xis[..., 0, :], h)
        for m in range(1, self.n_ctx):
            out = out + self.banks[m](xis[..., m, :], h)
        return out

    def analytical_grad(self, xis, h):
        out = self.banks[0].analytical_grad(xis[..., 0, :], h)
        for m in range(1, self.n_ctx):
            out = out + self.banks[m].analytical_grad(xis[..., m, :], h)
        return out

    def attractor_centres(self, xis):
        cs = [self.banks[m].attractor_centres(xis[..., m, :])
              for m in range(self.n_ctx)]
        return torch.cat(cs, dim=-2)


class AnisotropicDepthConditionedGaussianVTheta(nn.Module):

    def __init__(self, d, K, n_ctx, n_layers, rank=4,
                 w_scale=1.0, init_log_precision=None,
                 precision_max=None, force_norm_max=None,
                 code_init_std=0.02):
        super().__init__()
        self.d = d
        self.K = K
        self.n_ctx = n_ctx
        self.n_layers = n_layers
        self.bank = AnisotropicMultiContextGaussianVTheta(
            d=d, K=K, n_ctx=n_ctx, rank=rank, w_scale=w_scale,
            init_log_precision=init_log_precision,
            precision_max=precision_max,
            force_norm_max=force_norm_max,
        )
        self.depth_code = nn.Parameter(
            torch.randn(n_layers, n_ctx, d) * code_init_std
        )
        self._active_layer: int = 0

    @property
    def banks(self):
        return self.bank.banks

    def set_active_layer(self, layer_idx: int):
        self._active_layer = int(layer_idx)

    def _shift(self, xis):
        g = self._active_layer
        if not (0 <= g < self.n_layers):
            g = g % self.n_layers
        code = self.depth_code[g]
        lead = xis.dim() - 2
        code = code.view(*([1] * lead), self.n_ctx, self.d)
        return xis + code

    def forward(self, xis, h):
        return self.bank(self._shift(xis), h)

    def analytical_grad(self, xis, h):
        return self.bank.analytical_grad(self._shift(xis), h)

    def attractor_centres(self, xis):
        return self.bank.attractor_centres(self._shift(xis))


def install_aniso_depth_routing(model):
    vt = model.V_theta
    if not hasattr(vt, 'set_active_layer'):
        return

    _orig_layer_step = model._layer_step.__func__

    def _patched_layer_step(self, h, h_prev, m_b, gamma, layer_idx=0,
                            x_tokens=None, **kw):
        if hasattr(self.V_theta, 'set_active_layer'):
            self.V_theta.set_active_layer(layer_idx)
        return _orig_layer_step(self, h, h_prev, m_b, gamma,
                                layer_idx=layer_idx,
                                x_tokens=x_tokens, **kw)

    import types
    model._layer_step = types.MethodType(_patched_layer_step, model)


print('Anisotropic Gaussian V_theta classes defined.')

In [ ]:
# == Cell 4b: Fock-G1 First-Order Ablation Model ======================
#
# The ONLY change vs. the second-order FockMultiXiPARFLM is that the per-layer
# velocity memory delta = h - h_prev is forced to zero, collapsing the damped
# velocity-Verlet update to a pure first-order gradient step:
#
#   second order:  h_new = LN( h + delta/(1+dt*gamma) + dt^2/(m_b*(1+dt*gamma)) * f )
#   Fock-G1:       h_new = LN( h + beta * f ),  beta = dt^2/(m_b*(1+dt*gamma*))
#
# Achieved by overriding _fock_layer_step so it passes h_prev := h down to the
# inherited token dynamics, making delta identically zero at every layer. Every
# other channel (V_theta, V_phi, xi, register gates, reverse channel, fock-reg)
# is inherited unchanged, so the ablation isolates the inertial term only.

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig


class FockG1MultiXiPARFLM(FockMultiXiPARFLM):
    """First-order (gradient-flow) ablation of Fock v2.1 aniso-Gaussian.

    See companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_
    Order_Dynamics_Hypothesis.md section 6 (protocol "Fock-G1").
    """

    def _fock_layer_step(self, h, h_prev, r, salience, m_b, gamma, dt, layer_idx):
        # First order: discard the incoming velocity memory (h_prev := h) so
        # that delta = h - h_prev = 0 in the inherited damped-Verlet update.
        return super()._fock_layer_step(
            h, h, r, salience, m_b, gamma, dt, layer_idx,
        )


# -- Smoke test: first-order and second-order differ; FO output is finite --
def _fock_g1_smoke():
    _cfg = FockMultiXiPARFConfig(
        vocab_size=257, d=32, max_len=64, L=4, v_hidden=64, v_depth=2, dt=1.0,
        mass_mode='global', fixed_gamma=0.30, causal_force=True,
        ln_after_step=True,
        xi_channels=3, xi_alpha_inits=[0.5, 0.9, 0.99], xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive', top_k=4, score_head_hidden=8,
        gumbel_noise=False,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        fock_version='v2', n_registers=8,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True, d_k=16,
        tau_create_init=8.0,
        reverse_channel=True, per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, prefix_causal_registers=True,
    )
    torch.manual_seed(0)
    m2 = FockMultiXiPARFLM(_cfg)          # second order
    torch.manual_seed(0)
    m1 = FockG1MultiXiPARFLM(_cfg)        # first order
    m1.load_state_dict(m2.state_dict())   # identical weights

    x = torch.randint(0, 257, (2, 16))
    y = torch.randint(0, 257, (2, 16))

    m1.eval(); m2.eval()
    # The conservative force uses autograd.grad internally, so grad must be
    # enabled even for an eval-mode forward (mirrors evaluate()).
    with torch.enable_grad():
        l2, _ = m2(x, y)
        l1, _ = m1(x, y)
    l1 = l1.detach(); l2 = l2.detach()
    max_diff = (l1 - l2).abs().max().item()
    print(f'[Fock-G1 smoke] max|logit_1st - logit_2nd| = {max_diff:.4e} '
          f'(>0: dynamics diverge from layer 1 onward)')
    print(f'[Fock-G1 smoke] first-order logits finite: '
          f'{bool(torch.isfinite(l1).all())}')

    m1.train()
    _, loss1 = m1(x, y)
    loss1.backward()
    _n_grad = sum(1 for p in m1.parameters()
                  if p.grad is not None and p.grad.abs().sum() > 0)
    print(f'[Fock-G1 smoke] loss={loss1.item():.4f}  params_with_grad={_n_grad}')

    assert bool(torch.isfinite(l1).all()), 'first-order logits not finite'
    assert max_diff > 0, 'first- and second-order identical (delta not zeroed?)'


_fock_g1_smoke()
print('Fock-G1 first-order model OK')


In [ ]:
# == Cell 5: Model config + build + aniso V_theta swap =================
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

ARCH_TIERS = [
    (384, 16, 32),
    (384, 12, 16),
    (256, 16, 16),
    (256,  8, 16),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=FIXED_GAMMA,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        register_repulsion_kind=REGISTER_REPULSION_KIND,
        prefix_causal_registers=True,
    )


model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockG1MultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())

        _init_log_prec = -math.log(d)
        _prec_max = 2.0 / d
        mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
            d=d,
            K=V_THETA_WELLS_PER_HEAD,
            n_ctx=V_THETA_N_HEADS,
            n_layers=cfg.L,
            rank=ANISO_RANK,
            w_scale=W_SCALE,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        ).to(DEVICE)
        install_aniso_depth_routing(mdl)

        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} Aniso-Gaussian)')

        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} -- trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# -- Auto batch size (GRAD_ACCUM is fixed from Cell 0) --
BATCH_SIZE = 4
if DEVICE == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _probe_sizes = [16, 12, 8, 6, 4] if _vram_gb >= 70 else [12, 8, 6, 4]
    for bs in _probe_sizes:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())

d = model_cfg.d
print(f'\nModel: FockMultiXiPARFLM v2.1 + Anisotropic Gaussian V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: aniso-gaussian  rank={ANISO_RANK}  '
      f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
assert getattr(model, '_gamma_value', None) is not None, (
    'Fock-G1 requires fixed gamma (fixed_gamma set) so beta is a constant')
_gamma_star = float(model.gamma.item())
print(f'  dynamics: FIRST-ORDER (Fock-G1, delta:=0)  gamma*={_gamma_star:.3f} '
      f'(fixed => beta = dt^2/(m_b*(1+dt*gamma*)) constant)')

In [ ]:
# == Cell 6: Training loop =============================================

LR            = 3e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3

PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,
    'creation_gate': 0.3,
    'destruction_gate': 0.3,
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,
    'register': 0.3,
    'depth_code': 0.5,
}
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0
GRAD_SPIKE_COOLDOWN  = 0
EVAL_INTERVAL = 500
EVAL_ITERS    = 40
LOG_INTERVAL  = 50
CAUSAL_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_K = 256
TRAINED_LEAK_PROBE_PAIRS = 2

if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def fock_coupling_reg(mdl, lam, eps):
    alphas = mdl.xi_module.alpha
    return -lam * torch.log(alphas + eps).sum()


def forward_with_vreg(x, targets, lambda_v, lambda_fock, fock_eps):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    fock_reg_value = torch.tensor(0.0, device=x.device)
    loss = loss_ntp

    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss + lambda_v * v_reg_value

    if lambda_fock > 0:
        fock_reg_value = fock_coupling_reg(model, lambda_fock, fock_eps)
        loss = loss + fock_reg_value

    return loss, loss_ntp, v_reg_value, fock_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def run_causal_probe(step_num):
    import math as _math
    from model_gaussian_vtheta import (
        DepthConditionedMultiContextGaussianVTheta as _DCMCGVT,
        install_depth_routing as _idr,
    )
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=3, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockG1MultiXiPARFLM(_probe_cfg)
    _probe_model.V_theta = _DCMCGVT(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0/_PROBE_D, code_init_std=0.02,
    )
    _idr(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _max_delta = max(_max_delta,
                     float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item()))

    _passed = (_max_delta == 0.0)
    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} -- running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)
    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)
    model.train()

    result = {
        'step': step_num,
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'v_theta_depth_code_init_std': V_THETA_DEPTH_CODE_INIT_STD,
            'aniso_rank': ANISO_RANK,
            'lambda_fock_reg': LAMBDA_FOCK_REG,
            'first_order': FIRST_ORDER,
            'gamma_star': GAMMA_STAR,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': f'fock_g1_parf_multixi_v2.1_aniso_gaussian_dcvt{V_THETA_N_HEADS}',
        'corpus': 'openwebtext',
        'phase': 7,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# -- Optimizer --
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')

# -- Resume --
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# -- Training state --
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]


def _log_write(record_str):
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting...')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}')
                return


t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
run_fock_reg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0


def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s


steps_this_session = 0

# -- Schedule summary --
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'Fock-G1 (FIRST-ORDER) Aniso-Gaussian + Fock-Reg: steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')')
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF')
print(f'{"="*60}\n')


def _assign_clip_group(pname):
    low = pname.lower()
    for sub, thr in GRAD_CLIP_OVERRIDES.items():
        if sub.lower() in low:
            return f'override:{sub}', thr
    return pname.split('.', 1)[0], GRAD_CLIP


def per_group_grad_norms(mdl):
    groups = {}
    for n, p in mdl.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        key, _ = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
    out = {}
    for key, ps in groups.items():
        sq = 0.0
        for p in ps:
            sq += float(p.grad.detach().norm()) ** 2
        out[key] = sq ** 0.5
    return out


def clip_grads_per_group(mdl):
    groups, thr = {}, {}
    _dev = None
    for n, p in mdl.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        if _dev is None:
            _dev = p.grad.device
        key, mx = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
        thr[key] = mx
    total_sq = torch.zeros((), device=_dev) if _dev is not None else torch.zeros(())
    per_group = {}
    for key, ps in groups.items():
        gn = nn.utils.clip_grad_norm_(ps, thr[key])
        per_group[key] = float(gn)
        if key not in WATCHDOG_EXCLUDE_GROUPS:
            total_sq = total_sq + gn.detach() ** 2
    return total_sq.sqrt(), per_group


_last_pg_norms = {}
_last_spike_step = -10**9
for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    accum_fock_reg = 0.0
    accum_rep = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
            x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
        if REGISTER_REPULSION:
            _rep = model.pop_repulsion_loss()
            loss = loss + _rep
            accum_rep += float(_rep.detach()) / GRAD_ACCUM
        (loss / GRAD_ACCUM).backward()
        accum_ntp      += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg     += float(v_reg.detach()) / GRAD_ACCUM
        accum_fock_reg += float(fock_reg.detach()) / GRAD_ACCUM

    if GRAD_CENTRALIZATION:
        for p in model.parameters():
            if p.grad is not None and p.grad.dim() >= 2:
                p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

    if PER_GROUP_CLIP:
        grad_norm, _last_pg_norms = clip_grads_per_group(model)
    else:
        _last_pg_norms = per_group_grad_norms(model) if GRAD_SPIKE_DEBUG else {}
        if model.V_phi is not None:
            nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
        grad_norm = nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)

    if GRAD_SPIKE_DEBUG:
        _tot_preclip = float(grad_norm)
        if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
            _last_spike_step = step
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(),
                              key=lambda kv: kv[1], reverse=True)[:8]
                _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
            else:
                _brk = '(enable PER_GROUP_CLIP for breakdown)'
            print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                  f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}  fock_reg={accum_fock_reg:.4f}')
            print(f'[spike]   top groups: {_brk}')
            _log_write(json.dumps({
                'step': step + 1, 'event': 'grad_spike',
                'pre_clip_grad_norm': round(_tot_preclip, 2),
                'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                'fock_reg': round(accum_fock_reg, 4),
                'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
            }) + '\n')

    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        for bank in model.V_theta.banks:
            if hasattr(bank, 'clamp_params'):
                bank.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # -- Watchdog --
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        if _last_pg_norms:
            _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
            print('[watchdog] top group norms (pre-clip): '
                  + ', '.join(f'{k}={v:.1f}' for k, v in _top))
        _log_write(json.dumps({
            'step': step + 1, 'event': 'watchdog_reload',
            'ema_grad_norm': round(_grad_norm_ema, 2),
            'above_thresh_steps': _grad_norm_above_thresh,
        }) + '\n')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    run_fock_reg += accum_fock_reg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        avg_fock_reg = run_fock_reg / n_run
        run_ntp, run_vreg, run_fock_reg, n_run = 0.0, 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        _top_grp = ''
        if PER_GROUP_CLIP and _last_pg_norms:
            _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
            _top_grp = f'top[{_k}]={_v:.1f}  '
        _rep_str = f'rep={accum_rep:.4f}  ' if REGISTER_REPULSION else ''
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  fock_reg={avg_fock_reg:.4f}  '
            f'lr={lr_now:.2e}  grad={float(grad_norm):.2f}  {_rep_str}{_top_grp}'
            f'gamma={model.gamma.item():.3f}  alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)')
        _log_write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'fock_reg': avg_fock_reg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'reg_repulsion': accum_rep,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        _log_write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

    if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
        _cp_passed, _cp_delta = run_causal_probe(step + 1)
        _log_write(json.dumps({
            'step': step + 1,
            'causal_probe_passed': _cp_passed,
            'causal_probe_max_delta': _cp_delta,
        }) + '\n')

    if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
        _tlp_result = run_trained_leak_probe(step + 1)
        _log_write(json.dumps(_tlp_result) + '\n')

_log_fh[0].close()
print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')

## Fock v2.1 component diagnosticsStandalone probe -- safe to run any time against the live model or afreshly loaded checkpoint. It answers two questions:1. **Structural health** -- is each Fock piece being *used well*?2. **PPL attribution** -- how much does each piece actually *buy*?

In [ ]:
# == Cell 7: Component diagnostics =====================================
import torch
_bp = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
_bd = torch.load(_bp, map_location=DEVICE, weights_only=False)
model.load_state_dict(_bd['model_state_dict'], strict=False)
model.eval()
print(f"Probe target -> {_bp.name}  step {_bd.get('step')}  PPL {_bd.get('val_ppl'):.2f}")
del _bd
import gc; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

import gc, math, sys, torch, numpy as np
for _a in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _a): setattr(sys, _a, None)
model.zero_grad(set_to_none=True)
try: optim.zero_grad(set_to_none=True)
except Exception: pass
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU free {_free/1e9:.1f} / {_total/1e9:.1f} GB before probe')

PROBE_BS = 2
_rng = np.random.default_rng(1234)
def _mk(n, bs):
    return [(torch.from_numpy(a).to(DEVICE), torch.from_numpy(b).to(DEVICE))
            for a, b in (get_batch(val_ids, bs, BLOCK_SIZE, _rng) for _ in range(n))]
def _eval_on(batches):
    model.eval(); losses = []
    for x, y in batches:
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(float(loss.item()))
        del loss
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(losses))

# --- 1. structural health ---
model.eval(); model.set_fock_capture(True)
_hx, _hy = _mk(1, PROBE_BS)[0]
with torch.enable_grad():
    _out = model(_hx, _hy)
del _out
rep = model.fock_component_report()
_cols = ['layer','active_frac','reg_cos_sim','create_entropy','create_alpha_max',
         'rev_entropy','rev_scale','qforce_ratio','destroy_mean']
print('='*72); print('Fock v2.1 STRUCTURAL HEALTH'); print('='*72)
print('  '.join(f'{c[:10]:>10}' for c in _cols))
for dd in rep['per_layer']:
    print('  '.join(f'{str(dd.get(c)):>10}' if isinstance(dd.get(c),(bool,type(None)))
                    else f'{float(dd.get(c)):>10.3f}' for c in _cols))
print('-'*72); print('summary:', {k: round(v,3) for k,v in rep['summary'].items()})
for f in rep.get('flags', []): print('  * '+f)
del _hx, _hy, rep; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

# --- 2. PPL attribution ---
_pb = _mk(40, PROBE_BS)
base = _eval_on(_pb); base_ppl = math.exp(base); rows = [('full model', base)]
if getattr(model, 'reverse_channel_scale', None) is not None:
    _s = model.reverse_channel_scale.detach().clone()
    with torch.no_grad(): model.reverse_channel_scale.zero_()
    rows.append(('  - reverse channel', _eval_on(_pb)))
    with torch.no_grad(): model.reverse_channel_scale.copy_(_s)
_thr = model.cfg.register_salience_threshold
try:
    model.cfg.register_salience_threshold = 1e9
    rows.append(('  - registers (all)', _eval_on(_pb)))
finally:
    model.cfg.register_salience_threshold = _thr
print('\n'+'='*72)
print(f"{'arm':<22}{'loss':>10}{'ppl':>10}{'dPPL':>10}")
for n, l in rows:
    p = math.exp(l); print(f'{n:<22}{l:>10.4f}{p:>10.2f}{p-base_ppl:>+10.2f}')

In [ ]:
# == Cell 8: Training curve ============================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
alpha_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
                if 'xi_alphas' in e and 'event' not in e:
                    alpha_entries.append(e)
            except Exception:
                pass

if eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Fock-G1 (first-order) aniso r={ANISO_RANK} + fock-reg (OWT d=384)',
            linewidth=1.5, color='#C62828')
    ax.axhline(y=SECOND_ORDER_ANCHOR_PPL, color='green', linestyle='--', alpha=0.7,
               label=f'2nd-order anchor gamma*=0.10 ({SECOND_ORDER_ANCHOR_PPL})')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Fock-G1 (first-order) Aniso-Gaussian + Fock-Reg -- OWT d=384 (Phase 1)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if alpha_entries:
        ax = axes[1]
        a_steps = [e['step'] for e in alpha_entries]
        n_ch = len(alpha_entries[0]['xi_alphas'])
        for k in range(n_ch):
            ax.plot(a_steps, [e['xi_alphas'][k] for e in alpha_entries],
                    'o-', label=f'alpha_{k+1}', markersize=2, linewidth=1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Fock coupling strengths (alpha_k)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'training_curve_fock_g1_aniso_gaussian_owt.png', dpi=150)
    plt.show()
    print(f'Saved: {RESULTS_DIR / "training_curve_fock_g1_aniso_gaussian_owt.png"}')
else:
    print('No eval data to plot.')